<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/11-customer-craft/01-scoping-and-discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Craft: Scoping & Discovery

**Goal:** The half of the FDE job that isn't code: turning a vague customer ask into a scoped, buildable, *evaluable* system, and communicating the tradeoffs. This is a whole interview round, and the skill most technical candidates can't evidence.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Setup

The one interactive exercise below uses Groq (as a stand-in "customer" to practice discovery questions against). The two setup cells are all it needs.

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client.
2. **Load your API key.** Free key at [console.groq.com](https://console.groq.com/) (no credit card); in Colab add it via the **key icon** → **Add new secret** named exactly `GROQ_API_KEY`, notebook access on. Locally, set `GROQ_API_KEY` in your environment.

(Full walkthrough: [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

In [ ]:
from aien import setup

# Used only by the discovery role-play exercise below.
client, MODEL = setup()

## Why a code repo has a notebook about talking to people

Every other section made you a better *builder*. This one addresses why builders don't get the FDE offer: the role is **~60% product engineering, 25% applied-model judgment, 15% consulting**, and the 15% is where most backend engineers can't show evidence. The Forward Deployed Engineer title (Palantir's coinage, now core at OpenAI and Anthropic) literally means *deployed forward into the customer's problem*. The interview loop reflects it with a **customer scenario round**: role-play a discovery call or a broken deployment, scope a vague ask under questioning.

You can't fully learn this from a notebook. You learn it by doing it with real people (the exercises push you there). But you *can* learn the frameworks and the artifacts that make you credible, so the round isn't the one you walk in unprepared for. Three things this notebook gives you: a **discovery method**, a **scoping doc** template, and the **demo discipline** that reads as senior.

## The core skill: vague ask → scoped system

Customers don't hand you specs. They hand you *"we want AI for our support tickets."* That sentence hides a dozen unmade decisions, and your job is to surface them **before** you build, because the most expensive mistake in this role is building the wrong thing well.

The discipline is a small set of questions that turn fog into scope. Run every ask through them:

| Question | Why it's load-bearing |
|---|---|
| **What's the actual job to be done?** | "AI for support tickets" could be triage, drafting replies, summarizing, or deflection. Each is a different system. |
| **Who's the user, and what do they do today?** | The workflow you're inserting into determines everything. Watch them work if you can. |
| **What does success look like, as a number?** | "Better support" is unbuildable. "Cut first-response time 30%" or "deflect 20% of tier-1 tickets" is. This is your eval target (section 02–04). |
| **What's the cost of being wrong?** | A wrong summary is annoying; a wrong refund is expensive; a wrong medical answer is dangerous. Sets how much guardrail/eval/human-in-loop you need (sections 05, 07). |
| **What data exists, and can I see it?** | The messy reality of their data decides feasibility. "We have docs" often means a shared drive of PDFs from 2015. |
| **What's the smallest version that's useful?** | Ship the thinnest slice that solves a real slice of the problem. Scope creep kills FDE projects. |

Notice these map straight onto the rest of the curriculum: success-as-a-number is your eval, cost-of-wrong sets your guardrails, data-reality decides RAG feasibility. **Discovery is where you decide which of the skills you learned even apply.**

## Practice: interrogate a vague ask

Below, the model plays a non-technical stakeholder who wants "an AI assistant for our sales team." Practice discovery: ask it questions, and it answers in-character. Your goal isn't to get answers — it's to notice *which questions collapse the ambiguity fastest*. Edit `my_question` and re-run; try to reach a scoped one-sentence problem statement in under 8 questions.

In [ ]:
STAKEHOLDER = (
    "You are Dana, a non-technical VP of Sales at a mid-size B2B software company. "
    "You asked an engineer to build 'an AI assistant for our sales team.' You have "
    "only a fuzzy idea; you have NOT thought about specifics. Answer questions in "
    "character, briefly and realistically — reveal detail only when asked a good, "
    "specific question. If asked something vague, give a vague answer. Never design "
    "the system for them; make them extract it. Reality (reveal only if probed): reps "
    "waste ~2 hrs/day hunting for answers in scattered PDFs, Slack, and a CRM; the real "
    "pain is slow, wrong answers to prospect questions during live calls.")

conversation = []
def ask(question):
    conversation.append({"role": "user", "content": question})
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=180,
        messages=[{"role": "system", "content": STAKEHOLDER}] + conversation)
    a = resp.choices[0].message.content
    conversation.append({"role": "assistant", "content": a})
    print(f"YOU:  {question}\nDANA: {a}\n")

# Try a vague opener vs a sharp one and watch how much signal each returns.
ask("What do you want the AI assistant to do?")            # vague -> vague
ask("Walk me through what a sales rep does in a normal day where they get stuck.")  # sharp

Notice the difference: the open "what do you want" question gets a fog answer; the "walk me through where they get stuck" question surfaces the *actual* pain. Good discovery is mostly asking about the current workflow and the moment it breaks, not about "the AI." Keep going in the cell above until you can write the one-sentence problem statement below.

## The artifact: a one-page scoping doc

The output of discovery isn't a system. It's a **scoping doc** you can defend under hostile questioning (which is exactly the interview). One page, and every LLM project you build (including the capstone, section 12) should start with it. The template:

```
SCOPING DOC — <project name>

1. PROBLEM (1 sentence)
   Who has what pain, in what workflow. No mention of "AI".
   e.g. "Sales reps lose ~2 hrs/day to slow, unreliable answers to prospect
        questions during live calls, because knowledge is scattered across PDFs,
        Slack, and the CRM."

2. USER & CURRENT WORKFLOW
   Who uses it, what they do today, where it breaks.

3. SUCCESS METRIC (a number)
   How we'll know it worked. This is the eval target.
   e.g. "Rep can get a correct, sourced answer in <10s for 80% of prospect
        questions" — measured on a golden set of 50 real questions.

4. SCOPE — smallest useful version
   IN:  RAG over the product/pricing docs, answers with citations, Slack bot.
   OUT: CRM writes, multi-language, voice. (Named exclusions prevent creep.)

5. COST OF BEING WRONG → safeguards
   Wrong answer on a live call = lost deal. So: citations required, "I don't
   know" over guessing, human-visible confidence. (Drives eval + guardrails.)

6. DATA
   What exists, its state, access. "~200 PDFs + CRM export; messy, no schema."

7. RISKS / OPEN QUESTIONS
   The things that could sink it, named honestly.
```

The senior move is section 5: naming the cost of being wrong and letting it drive the engineering. Junior candidates jump to architecture; senior ones scope the failure surface first.

## The demo, and the credibility move

You'll demo the thing you built, both to the customer and in the interview. Two rules separate a demo that lands from one that doesn't:

1. **Structure: problem → live walkthrough → what it can't do.** Open with the pain you're solving (from your scoping doc), show it working on a *real* example, then (crucially) state its limits.

> **💡 Why it matters —** saying what it *can't* do is the credibility move. Everyone's seen the impressive-but-brittle AI demo; the engineer who says *"it declines questions outside the docs, and here's one it gets wrong, and why"* is the one trusted with production. Confidence about limitations reads as senior; over-claiming reads as junior. (Same instinct as the "I don't know" behavior you built into RAG in section 03.)

Prepare two **war stories** too: times you navigated ambiguity or a difficult stakeholder. FDE behavioral rounds weight these as heavily as the technical ones.

## Exercises (do these with real people; that's the point)

1. **Free consult.** Offer two colleagues or small-business contacts a free 30-minute "AI consult." Run the discovery questions from this notebook. Write a one-page scoping doc for each. This is the single highest-value exercise in the repo for FDE roles: real ambiguity, real scoping, real artifact.
2. **Scope your capstone.** Before building the section-12 capstone, write its scoping doc using the template above. If you can't fill in a numeric success metric, you're not ready to build — go back to discovery.
3. **Rehearse the demo.** Record a 10-minute demo of any project you've built: problem, live walkthrough, what it can't do. Watch it back. Did you state limitations? Did you lead with the user's problem or with your architecture?
4. **Two war stories.** Write two STAR-format stories where you navigated ambiguity or a hard stakeholder. These are your behavioral-round answers; having them written beats improvising.
5. **Interrogate the stakeholder bot further.** In the role-play cell, try to reach a defensible one-sentence problem statement in the fewest questions. Then change the `STAKEHOLDER` system prompt to a different vague ask ("AI for our HR team") and do it again. Discovery is a repeatable method, not a script.